# 00 - Data validation, database and the ten analytical queries
PRD Week 1: load and validate the four CSVs, build the relational schema with enforced keys/constraints, run and interpret the ten SQL queries.

**Engine note:** PostgreSQL is not available in the build sandbox. The schema DDL is taken *verbatim* from `sql/credit_risk_platform.sql` (sections 1-5) and executed on SQLite with foreign keys ON; the ten queries are run from the same file after a mechanical translation of three PostgreSQL-only constructs (`DATE_TRUNC`, `TO_CHAR`, `PERCENTILE_CONT`). Running the original file on PostgreSQL is still your step.

In [1]:
import sys, pathlib
sys.path.insert(0, str(pathlib.Path.cwd().parent))
import warnings; warnings.filterwarnings('ignore')
import numpy as np, pandas as pd, matplotlib.pyplot as plt, seaborn as sns, json
sns.set_theme(style='whitegrid')
pd.set_option('display.width', 200); pd.set_option('display.max_columns', 40)
from src import data_load as dl
raw = dl.load_raw()
pd.DataFrame({k: {'rows': len(v), 'cols': v.shape[1], 'missing_cells': int(v.isna().sum().sum()), 'exact_duplicates': int(v.duplicated().sum())} for k, v in raw.items()}).T

,rows,cols,missing_cells,exact_duplicates
customers,5000,20,0,0
credit_history,5000,14,0,0
loan_applications,7500,17,4270,0
transactions,40079,8,0,0


In [2]:
v0 = dl.validate(raw)
print(int(v0.passed.sum()), '/', len(v0), 'checks pass on the RAW files')
v0[~v0.passed]

22 / 23 checks pass on the RAW files


,check,passed,detail
13,customers.email is unique (schema has UNIQUE),False,duplicate emails=2


**Finding:** the raw files are tidy (no nulls outside the two legitimately-empty columns, no duplicates, no orphan keys, all PRD value rules hold) - with **one blocking exception**: two e-mail addresses are shared by two *different* customers each. The schema declares `email UNIQUE`, so a PostgreSQL `\copy` of the raw file would fail on those rows. Fix: keep the first customer's e-mail, set the later one to NULL (e-mail is excluded from modelling anyway) and load the cleaned file.

In [3]:
t, log = dl.clean(raw)
print(log.to_string(index=False))
v = dl.validate(t); print(int(v.passed.sum()), '/', len(v), 'checks pass after cleaning')
v[v.check.str.startswith('INFO')]

    table                                                      issue  rows                                              action                   ids
customers e-mail shared by two different customers (violates UNIQUE)     2 kept first customer's e-mail, set later one to NULL CUST104628,CUST104915
23 / 23 checks pass after cleaning


,check,passed,detail
21,INFO applications dated before customer_since,True,935 of 7500 - kept; documented in data_dictionary.md
22,INFO bureau snapshot date is after most applications,True,"as_of_date=['2024-06-30'], applications 2022-07-01..2024-06-29 - temporal-leakage risk, see model card"


In [4]:
con = dl.build_sqlite(tables=t)
print({tb: con.execute(f'SELECT COUNT(*) FROM {tb}').fetchone()[0] for tb in dl.TABLES})
# do the constraints really bite?
import sqlite3
for s in ["INSERT INTO credit_history (credit_id,customer_id,as_of_date,credit_score) VALUES ('X','CUST100001','2024-06-30',950)", "INSERT INTO transactions (transaction_id,customer_id,transaction_date,transaction_type,amount) VALUES ('T','NOPE','2024-01-01','Debit',5)", "INSERT INTO loan_applications (application_id,customer_id,application_date,loan_type,approval_status) VALUES ('A','CUST100001','2024-01-01','Home','Maybe')"]:
    try: con.execute(s); print('ACCEPTED (bad!)')
    except sqlite3.IntegrityError as e: print('rejected ->', e)

{'customers': 5000, 'credit_history': 5000, 'loan_applications': 7500, 'transactions': 40079}
rejected -> CHECK constraint failed: credit_score BETWEEN 300 AND 900
rejected -> FOREIGN KEY constraint failed
rejected -> CHECK constraint failed: approval_status IN ('Approved','Rejected','Under Review')


## The ten analytical queries (executed from the PostgreSQL file)

In [5]:
res = dl.run_analytics(con)
print({k: (len(r['df']) if r['df'] is not None else r['error']) for k, r in res.items()})
for k in ['7.1', '7.5']:
    print(k, res[k]['title']); print(res[k]['df'].to_string(index=False)); print()

{'7.1': 5, '7.2': 24, '7.3': 14, '7.4': 100, '7.5': 6, '7.6': 6, '7.7': 24, '7.8': 3, '7.9': 50, '7.10': 4}
7.1 Credit score analysis: distribution across standard risk bands
        credit_band  customers  avg_score  avg_utilization  avg_late_30d
Excellent (750-900)        571      787.0            0.270          0.39
     Good (700-749)        822      722.0            0.319          0.38
     Fair (650-699)       1164      674.0            0.349          0.43
     Poor (580-649)       1440      618.0            0.393          2.22
Very Poor (300-579)       1003      526.0            0.450          2.22

7.5 Default rate analysis by loan type (disbursed loans only)
loan_type  disbursed_loans  defaults  default_rate_pct  avg_interest_rate
     Home              326       103             31.60               9.20
 Business              290        86             29.66              15.95
Education              336        92             27.38              10.96
     Gold              386  

In [6]:
for k in ['7.8', '7.10']:
    print(k, res[k]['title']); print(res[k]['df'].to_string(index=False)); print()

7.8 Risk categorisation blending bureau score and utilisation
risk_category  customers  avg_score
     Low Risk        992      751.0
  Medium Risk       2268      674.0
    High Risk       1740      558.0

7.10 Top predictors of loan default: default rate sliced by driver
          driver  default_rate_pct    n
credit_score<600             40.11  379
 utilization>0.8             33.33   36
prior_default>=1             33.10  142
         dti>0.5             28.50 2340



### Reading the results critically
* **7.1/7.8** Two different 'risk' definitions (score-only bands vs score + utilisation) give different groupings - neither is a model; they are rule-based baselines.
* **7.5** Default rates differ by loan type (Home and Business highest), but each loan type has only between 290 and 1,395 disbursed loans, so differences of a few points are within noise.
* **7.10** The rule-slice table is a *univariate* view: `credit_score<600` and `prior_default>=1` overlap heavily, so these slices cannot be added up. `dti>0.5` is the broadest and the strongest single driver, which the model later confirms.

In [7]:
# independent cross-check of two SQL answers with pandas
l = t['loan_applications']; h = t['credit_history']
m = l.merge(h, on='customer_id').dropna(subset=['loan_default'])
print('7.5 Home default rate  SQL', res['7.5']['df'].set_index('loan_type').loc['Home', 'default_rate_pct'], '| pandas', round(100 * l[(l.loan_type == 'Home')].loan_default.mean(), 2))
print('7.10 credit_score<600  SQL', res['7.10']['df'].set_index('driver').loc['credit_score<600', 'default_rate_pct'], '| pandas', round(100 * m[m.credit_score < 600].loan_default.mean(), 2))

7.5 Home default rate  SQL 31.6 | pandas 31.6
7.10 credit_score<600  SQL 40.11 | pandas 40.11
